In [1]:
import os # for file IO and stuff related to dataset
import matplotlib.pyplot as plt # for plotting  
import pandas as pd # out goto guy for processing CSVs and many more things
from IPython.display import Image, display  # Added for viewing in notebook
import numpy as np # for running interim data conversions
import wfdb  # for reading the ECG samples

In [2]:
annotations_processed_csv =  pd.read_csv("C:\Project\Research-Project\DataFrame_Extracts\processed_annotations.csv")
annotations_processed_csv.head()

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units
0,A00001,N,C:\Project\Research-Project\training2017\\A00001,300,9000,1,mV
1,A00002,N,C:\Project\Research-Project\training2017\\A00002,300,9000,1,mV
2,A00003,N,C:\Project\Research-Project\training2017\\A00003,300,18000,1,mV
3,A00004,A,C:\Project\Research-Project\training2017\\A00004,300,9000,1,mV
4,A00005,A,C:\Project\Research-Project\training2017\\A00005,300,18000,1,mV


In [3]:
# selecting all samples with more then 9k records
annotations_processed_9K_samples = annotations_processed_csv.loc[annotations_processed_csv.Signal_Length_Samples>=9000 ]

In [4]:
annotations_processed_9K_samples

,SampleID,Inference,Matlab_files,Sampling_Rate_Hz,Signal_Length_Samples,Num_Channels,Units
0,A00001,N,C:\Project\Research-Project\training2017\\A00001,300,9000,1,mV
1,A00002,N,C:\Project\Research-Project\training2017\\A00002,300,9000,1,mV
2,A00003,N,C:\Project\Research-Project\training2017\\A00003,300,18000,1,mV
3,A00004,A,C:\Project\Research-Project\training2017\\A00004,300,9000,1,mV
4,A00005,A,C:\Project\Research-Project\training2017\\A00005,300,18000,1,mV
...,...,...,...,...,...,...,...
8523,A08524,N,C:\Project\Research-Project\training2017\\A08524,300,9000,1,mV
8524,A08525,O,C:\Project\Research-Project\training2017\\A08525,300,9000,1,mV
8525,A08526,N,C:\Project\Research-Project\training2017\\A08526,300,10904,1,mV
8526,A08527,N,C:\Project\Research-Project\training2017\\A08527,300,9000,1,mV


In [5]:
from sklearn.model_selection import StratifiedShuffleSplit
# we're using StratifiedShuffleSplit to balance the classes, if we use a regular train-test split the resultant split might be a tad unbalanced 

In [6]:
#this function takes care of the balanced split 
def split_dataset_stratified(df, train_ratio=0.8, random_state=42):
    """
    Function 1 (Corrected): A regular, stratified train-test split.
    Because every record is still a single row at this point, this balances
    the medical classes perfectly across Train and Test without any leakage.
    """
    # Initialize the stratified splitter for a single split
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=train_ratio, random_state=random_state)
    
    # Split using the 'Inference' column to keep class distributions identical
    for train_idx, test_idx in splitter.split(df, df['Inference']):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
    
    return train_df, test_df

In [7]:
# this is just the raw validation split
raw_train, raw_test = split_dataset_stratified(annotations_processed_9K_samples, train_ratio=0.8)

This is the piece-de-resistance of this run, splitting long records to make more records for trainning

In [8]:
def slice_ecg_records(df, set_name="Dataset"):
    """
    Function 2: Slices records based on lengths and logs detailed dataset statistics,
    including dropped/split counts and class size transformations.
    """
    sliced_rows = []
    
    # Trackers for logging
    total_input = len(df)
    dropped_count = 0
    single_chunk_count = 0
    split_count = 0
    
    print(f"\n======== LOGGING STATISTICS FOR: {set_name.upper()} ========")
    print(f"Initial total records: {total_input}")
    
    # Capture initial class distribution (Inference column)
    initial_classes = df['Inference'].value_counts()
    
    for _, row in df.iterrows():
        pid = row['SampleID']
        length = int(row['Signal_Length_Samples'])
        
        # Rule 1: Less than 9k -> Ignore
        if length < 9000:
            dropped_count += 1
            continue
            
        # Rule 3: 13.5k to 18k -> Split into 2 samples (Head & Tail)
        if length >= 13500:
            split_count += 1
            
            # Record 1: Head window
            s1 = row.to_dict()
            s1['Sliced_SampleID'] = f"{pid}_1"
            s1['Start_Idx'] = 0
            s1['End_Idx'] = 9000
            sliced_rows.append(s1)
            
            # Record 2: Tail window
            s2 = row.to_dict()
            s2['Sliced_SampleID'] = f"{pid}_2"
            s2['Start_Idx'] = length - 9000
            s2['End_Idx'] = length
            sliced_rows.append(s2)
            
        # Rule 2: 9k to 13.5k -> Take first 9k points
        else:
            single_chunk_count += 1
            s_fixed = row.to_dict()
            s_fixed['Sliced_SampleID'] = f"{pid}_1"
            s_fixed['Start_Idx'] = 0
            s_fixed['End_Idx'] = 9000
            sliced_rows.append(s_fixed)
            
    # Convert to DataFrame to calculate final stats
    sliced_df = pd.DataFrame(sliced_rows)
    total_output = len(sliced_df) if not sliced_df.empty else 0
    
    # Print processing logs
    print(f"Dropped records (< 9k): {dropped_count}")
    print(f"Standard records (9k - 13.5k): {single_chunk_count}")
    print(f"Split records (>= 13.5k): {split_count} (Generated {split_count * 2} samples)")
    print(f"Final sliced total records: {total_output}")
    print("-" * 45)
    
    # Print class transformation logs
    print("CLASS SIZE TRANSFORMATION (Initial -> Sliced):")
    if not sliced_df.empty:
        final_classes = sliced_df['Inference'].value_counts()
        # Combine initial and final for a side-by-side view
        all_classes = sorted(list(set(initial_classes.index) | set(final_classes.index)))
        for cls in all_classes:
            init_val = initial_classes.get(cls, 0)
            final_val = final_classes.get(cls, 0)
            print(f"  * Class [{cls}]: {init_val} records -> {final_val} samples")
    else:
        print("  No samples generated.")
    print("=" * 45)
    
    return sliced_df


In [9]:
grown_train_set = slice_ecg_records(raw_train, set_name="Train Set")
grown_test_set = slice_ecg_records(raw_test, set_name="Test Set")


======== LOGGING STATISTICS FOR: TRAIN SET ========
Initial total records: 6048
Dropped records (< 9k): 0
Standard records (9k - 13.5k): 5226
Split records (>= 13.5k): 822 (Generated 1644 samples)
Final sliced total records: 6870
---------------------------------------------
CLASS SIZE TRANSFORMATION (Initial -> Sliced):
  * Class [A]: 500 records -> 580 samples
  * Class [N]: 3623 records -> 4026 samples
  * Class [O]: 1809 records -> 2138 samples
  * Class [~]: 116 records -> 126 samples

======== LOGGING STATISTICS FOR: TEST SET ========
Initial total records: 1513
Dropped records (< 9k): 0
Standard records (9k - 13.5k): 1295
Split records (>= 13.5k): 218 (Generated 436 samples)
Final sliced total records: 1731
---------------------------------------------
CLASS SIZE TRANSFORMATION (Initial -> Sliced):
  * Class [A]: 125 records -> 147 samples
  * Class [N]: 906 records -> 1017 samples
  * Class [O]: 453 records -> 538 samples
  * Class [~]: 29 records -> 29 samples


In [10]:
# Earlier we had this but since we've grown the dataset, we have to manually assign the X_train, y_train, X_test, y_test datasets
#X = Sample_wtih_9k_points.drop(columns=["Inference"])
#Y = Sample_wtih_9k_points["Inference"]


X_train = grown_train_set.drop(columns=["Inference"])
y_train = grown_train_set["Inference"]
X_test  = grown_test_set.drop(columns=["Inference"])
y_test  = grown_test_set["Inference"]
print(f" X_train shape: {X_train.shape} , X_test shape:{X_test.shape}, \n  y_train shape:{y_train.shape}, y_test shape:{y_test.shape}")

 X_train shape: (6870, 9) , X_test shape:(1731, 9), 
  y_train shape:(6870,), y_test shape:(1731,)


    function that Reads a pre-filtered ECG record that is known to be exactly 9000 samples.
    Returns a pristine (9000, 1) float32 numpy array ready for Conv1D.

In [11]:
#we took this function from our previous model, we gotta change it as now we're feeding the model out newely created d

def get_exact_1d_ecg_input(matlab_filepath, start_idx, end_idx):
    """
    Reads a raw MATLAB/WFDB signal from disk and cuts it precisely 
    between start_idx and end_idx using coordinates from the slicing dataframe.
    """
    # Clean file extensions if present
    record_path = str(matlab_filepath).replace('.mat', '').replace('.hea', '')
    
    try:
        # 1. Read raw voltage signals
        record = wfdb.rdrecord(record_path)
        signal = record.p_signal 
        
        # Pull the primary single lead channel (Lead I)
        lead_data = signal[:, 0]
        
        # 2. Slice the signal dynamically using your pipeline's markers
        # Explicit conversion to integers prevents indexing bugs
        sliced_data = lead_data[int(start_idx) : int(end_idx)]
        
        # 3. Safety Check: Verify the sliced segment is exactly 9000 points
        if len(sliced_data) != 9000:
            raise ValueError(
                f"Sliced length is {len(sliced_data)}, expected exactly 9000. "
                f"(Total file length: {len(lead_data)}, Indices: {start_idx} to {end_idx})"
            )
            
        # 4. Add the required Keras channel dimension and cast to float32
        # Transforms shape from (9000,) to (9000, 1)
        cnn_input_1d = np.expand_dims(sliced_data, axis=-1).astype('float32')
        
        return cnn_input_1d

    except Exception as e:
        print(f"🚨 Skipping record {record_path} due to error: {e}")
        return None

We will now build the base model

    # Input shape: 9000 time-series points, 1 single signal lead
    first convolution later
    **filters=32**,     # industry standard - but we can furyher downsize it to 16 and see if the model's prediction ability is still the same
                    # the idea behind was 32 filteres for 2D image so 16 filters for 1D image should be enough 

    **kernel_size=6**,  # Typical ECG has 0.8 seconds for a beat.
                        # 9000 samples represent 30 seconds of ECG; so 1 second → 300 samples; 1 ms = 300/1000 = 0.3 data points. 
                        # 1 full cycle is typically 0.8 seconds
                        # so 30 seconds will have 37.5 ish beats
                        # so 1 beat in the samples will have 9000/37.5 = 240 data-points
                        # The fastest transitions are for the QRS interval - typically 70-110 ms  (Q wave → R wave); (R wave → T wave)  
                        # so 70 ms = 70x(samples for 1 ms) = 70x0.3 = 21 samples. similarly 110 ms =110x0.3 = 33 samples
                        # BUT 
                        # A normal, sharp Q → R upstroke typically takes only 20-30 ms
                        # so, 20x0.3 = 6 samples, 30x0.3 = 9 samples.
                    # choosing the lower value so that the model can see the traisition,we choose conv. kernel filter size as 6                    

    **activation='relu'** # after the convolver takes 6 samples, multiplies and sums them up, the ReLU just checks if the value is positive or negative. this helps in speeding up the process as opposed to tanh(x) or sigmoid(x) which is heavy on the math.

    **input_shape**=(9000, 1) -  we had pre-choosen the samples with size=9000 points
                  
                  
    layers.MaxPooling1D(pool_size=2) - so that the learned features stay sharp and propogate down the line. if this was avg pooling, then, the work done by previous layers woud have been undone straightaway

    

In [12]:
from tensorflow.keras import layers, models

model_1d = models.Sequential([
    # Block 1: Catching raw features
    layers.Conv1D(filters=32, kernel_size=7, activation='relu', input_shape=(9000, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=4), # Downsample aggressive early on
    
    # Block 2: Middle-level patterns
    layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=4),
    
    # Block 3: Deep contextual patterns
    layers.Conv1D(filters=128, kernel_size=3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=4), # Your sequence is now compressed perfectly
    
    # The Global Bottleneck
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.5),

    # Classifier Head
    layers.Dense(64, activation='relu'),
    layers.Dense(4, activation='softmax')
])

model_1d.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 8994, 32)          256       
                                                                 
 batch_normalization (BatchN  (None, 8994, 32)         128       
 ormalization)                                                   
                                                                 
 max_pooling1d (MaxPooling1D  (None, 2248, 32)         0         
 )                                                               
                                                                 
 conv1d_1 (Conv1D)           (None, 2244, 64)          10304     
                                                                 
 batch_normalization_1 (Batc  (None, 2244, 64)         256       
 hNormalization)                                                 
                                                        

In [13]:
model_1d.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 8994, 32)          256       
                                                                 
 batch_normalization (BatchN  (None, 8994, 32)         128       
 ormalization)                                                   
                                                                 
 max_pooling1d (MaxPooling1D  (None, 2248, 32)         0         
 )                                                               
                                                                 
 conv1d_1 (Conv1D)           (None, 2244, 64)          10304     
                                                                 
 batch_normalization_1 (Batc  (None, 2244, 64)         256       
 hNormalization)                                                 
                                                        

In [22]:
def data_generator(df, path_col, label_col):
    label_mapping = {'N': 0, 'A': 1, 'O': 2, '~': 3}
    
    # Notice: No shuffling here anymore! It just reads row-by-row.
    for _, row in df.iterrows():
        file_path = row[path_col]
        raw_label = row[label_col]  
        
        label = label_mapping.get(raw_label, -1)
        if label == -1:
            continue  # Skip unexpected classes safely
            
        start_w = row['Start_Idx']
        end_w = row['End_Idx']
        
        ecg_array = get_exact_1d_ecg_input(file_path, start_w, end_w) 
        if ecg_array is None:
            continue
            
        yield ecg_array, label        


In [23]:
import tensorflow as tf
 
def create_ecg_dataset(df, path_col, label_col, batch_size=32, shuffle=True):
    
    # Shuffling happens ONLY here, controlled completely by the 'shuffle' flag
    current_df = df.sample(frac=1).reset_index(drop=True) if shuffle else df

    dataset = tf.data.Dataset.from_generator(
        generator=lambda: data_generator(current_df, path_col, label_col),
        output_signature=(
            tf.TensorSpec(shape=(9000, 1), dtype=tf.float32), 
            tf.TensorSpec(shape=(), dtype=tf.int32)           
        )
    )
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return dataset

In [20]:
model_1d.compile( optimizer='adam', 
               loss='sparse_categorical_crossentropy', 
               metrics=['accuracy'])


In [25]:
# Create your training pipeline (shuffled, batched, ready to learn)
train_dataset = create_ecg_dataset(grown_train_set, 'Matlab_files', 'Inference', batch_size=32, shuffle=True)

# Create your validation pipeline (ordered, consistent, perfect for evaluating)
val_dataset = create_ecg_dataset(grown_test_set, 'Matlab_files', 'Inference', batch_size=32, shuffle=False)


In [26]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Set up the auto-save configuration so taht in the event of a readbump, we don't loose 2+ hours of trainning

checkpoint_File = 'C:\Project\Research-Project\Models\ecg_model_checkpoint_V1.1.3.h5' 
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_File,             # The file path where it saves to disk
    monitor='val_loss',                   # Look at validation loss to judge performance
    save_best_only=True,                  # ONLY overwrite the file if the model improved!
    mode='min',                           # We want the lowest possible validation loss
    verbose=1                             # Print a message whenever it auto-saves
)

In [ ]:
import os
from tensorflow.keras.models import load_model

checkpoint_path = checkpoint_File

# Initialize a tracker variable
starting_epoch = 0

if os.path.exists(checkpoint_path):
    print(f" Found existing training checkpoint at '{checkpoint_path}'.")
    print("Resuming training from the last saved best state...")
    
    model_1d = load_model(checkpoint_path)
    
    # OPTIONAL FIX: If your checkpoint name saves the epoch number 
    # (e.g., "model_epoch_25.h5"), you can extract it here. 
    # Otherwise, you can hardcode or guess your starting point:
    starting_epoch = 25  

else:
    print(f" No checkpoint found at '{checkpoint_path}'.")
    print("Starting training fresh from Epoch 1...")
    
    # --- YOUR ARCHITECTURE GOES HERE ---
    # model_1d = build_your_cnn_model()
    # model_1d.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


# 2. Kick off the training engine (With Epoch Tracking Enabled)
history = model_1d.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    initial_epoch=starting_epoch,  # <-- CRITICAL ADDITION
    callbacks=[checkpoint_callback],  
    verbose=1
)


 No checkpoint found at 'C:\Project\Research-Project\Models\ecg_model_checkpoint_V1.1.3.h5'.
Starting training fresh from Epoch 1...
Epoch 1/50
     18/Unknown - 32s 2s/step - loss: 1.2692 - accuracy: 0.5104